# House Price Prediction — Classical Machine Learning
## Project 01: Linear Regression from Scratch Workflow

Welcome to our first Machine Learning project! In this notebook, we follow an end-to-end ML lifecycle:
1. **Phase 1**: Environment & Library Setup
2. **Phase 2**: Understand, explore, clean, and visualize housing data
3. **Phase 3**: Build and train a Scikit-Learn Linear Regression model
4. **Phase 4**: Evaluate performance using MAE, MSE, RMSE, and $R^2$ score
5. **Phase 5**: Serialize and persist the trained model using Joblib for production inference

### Phase 1 — Import Libraries
We import NumPy for numerical operations, Pandas for dataframe processing, Matplotlib for data visualization, and Scikit-Learn for model building and evaluation.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("All ML libraries imported successfully!")

### Phase 2 — Understand and Prepare Data
Now we load the house prices dataset from `../data/house_prices.csv` using Pandas.

In [ ]:
# Define dataset path
data_path = Path('../data/house_prices.csv')
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("First 5 Rows:")
df.head()

In [ ]:
# Dataset structural information
print("Dataset Summary Information:")
df.info()

print("\nDescriptive Statistics:")
df.describe()

#### Handling Missing Values
Real-world data often has null or missing entries. Let's inspect and impute missing values.

In [ ]:
# Check missing values per column
print("Missing values per feature:")
print(df.isnull().sum())

# Impute missing values with column medians
for col in df.columns:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed missing values in '{col}' with median: {median_val}")

print("\nRemaining missing values:", df.isnull().sum().sum())

#### Data Visualization — Relationship between Area and Price
Let's create a scatter plot of House Area (sq ft) vs House Price ($) to inspect the linear correlation.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df['area'], df['price'], color='#2563eb', alpha=0.7, edgecolors='b', s=50)
plt.title('House Area vs. Price (Exploratory Data Analysis)', fontsize=14, fontweight='bold')
plt.xlabel('Area (Square Feet)', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

correlation = df['area'].corr(df['price'])
print(f"Pearson Correlation between Area and Price: {correlation:.4f} (Strong positive correlation!)")

### Phase 3 — Build and Train the Model
We split our data into training (80%) and testing (20%) sets. Then we initialize `LinearRegression()` and train it using `model.fit()`.

In [ ]:
# Feature matrix X (2D) and target vector y (1D)
X = df[['area']]
y = df['price']

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

In [ ]:
# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

slope = float(model.coef_[0])
intercept = float(model.intercept_)

print("=== Model Learned Parameters ===")
print(f"Weight / Coefficient (Slope w):  ${slope:,.2f} per sqft")
print(f"Bias / Intercept (b):            ${intercept:,.2f}")
print(f"\nMathematical Formulation:")
print(f"Predicted Price = (${slope:.2f} * Area) + ${intercept:.2f}")

#### Visualizing the Learned Regression Line
Let's plot the fitted regression line against the training points to see how the model captures the trend.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(X_train, y_train, color='#3b82f6', label='Training Data', alpha=0.6)
line_x = np.linspace(X['area'].min(), X['area'].max(), 100).reshape(-1, 1)
line_y = model.predict(line_x)
plt.plot(line_x, line_y, color='#ef4444', linewidth=2.5, label=f'Best Fit Line: y = {slope:.1f}x + {intercept:,.0f}')
plt.title('Fitted Linear Regression Model', fontsize=14, fontweight='bold')
plt.xlabel('Area (Square Feet)', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

### Phase 4 — Model Evaluation
We evaluate the model on the unseen test set using standard regression metrics:
- **MAE (Mean Absolute Error)**: Average dollar error per prediction.
- **MSE (Mean Squared Error)**: Penalizes larger prediction errors heavily.
- **RMSE (Root Mean Squared Error)**: Square root of MSE, in the same dollar units as the target.
- **$R^2$ (Coefficient of Determination)**: Proportion of variance explained (1.0 = perfect fit).

In [ ]:
# Predict on test set
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Test Set Evaluation Metrics ===")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Mean Squared Error (MSE):  {mse:,.2f}")
print(f"Root Mean Squared (RMSE):  ${rmse:,.2f}")
print(f"R² Score:                  {r2:.4f} ({r2*100:.2f}% variance explained)")

In [ ]:
# Comparison Table of Actual vs Predicted
results_df = pd.DataFrame({
    'Area (sqft)': X_test['area'].values,
    'Actual Price ($)': y_test.values,
    'Predicted Price ($)': np.round(y_pred, 2),
    'Absolute Error ($)': np.round(np.abs(y_test.values - y_pred), 2)
})
results_df.head(10)

#### Understanding Overfitting vs. Underfitting
- **Underfitting**: If the model was too simple (e.g. constant prediction), it would fail on both training and test data (low $R^2$ on both).
- **Overfitting**: If a high-degree polynomial model memorized every training noise point, training error would be near zero, but test error would explode.
- **Good Generalization**: Here our Linear Regression achieves an $R^2 > 0.95$ on unseen test data, proving excellent generalizability without overfitting!

### Phase 5 — Model Persistence with Joblib
We save the trained Scikit-Learn model to disk so our FastAPI backend can load and serve it without needing to retrain.

In [ ]:
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)
model_save_path = models_dir / 'house_price_model.pkl'

# Save model
joblib.dump(model, model_save_path)
print(f"Model successfully saved to: {model_save_path.resolve()}")

In [ ]:
# Independent Reload Verification Test
loaded_model = joblib.load(model_save_path)
print("Model loaded from disk independently!")

# Test sample predictions for real estate inquiry
sample_areas = [1200, 1850, 2400, 3500]
print("\n--- Sample Predictions from Loaded Model ---")
for area in sample_areas:
    prediction = loaded_model.predict([[area]])[0]
    print(f"Area: {area:,} sqft  ==>  Estimated Price: ${prediction:,.2f}")